# TUGAS MANDIRI — PERTEMUAN 7
### Studi Kasus Terintegrasi: Analisis Big Data End-to-End

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 7 |
| **Topik** | Studi Kasus Terintegrasi: Kopi Nusantara |
| **Nama Mahasiswa** | *(Isi Nama Anda)* |
| **NPM** | *(Isi NPM Anda)* |

---
## Skenario Tugas
Sebagai data analyst jaringan kedai kopi **"Kopi Nusantara"**, Anda diminta menyusun laporan performa triwulan yang mencakup penjualan per cabang, menu terlaris via Window Function, kontribusi barista via Spark SQL, dan penyimpanan laporan final ke HDFS/Parquet.


### Persiapan Dataset (3 Sumber Data)


In [ ]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# 1. Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# 2. Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# 3. Sumber 3: Data transaksi (CSV) — 4000 baris
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)
print("Data Kopi Nusantara berhasil digenerate.")


### Bagian A: EXTRACT — Membaca Ketiga Sumber Data


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg, row_number, desc
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_cabang = spark.read.json("tugas7_cabang.json")
df_menu = spark.read.csv("tugas7_menu.csv", header=True, inferSchema=True)
df_transaksi = spark.read.csv("tugas7_transaksi.csv", header=True, inferSchema=True)

print("Jumlah Baris: Cabang =", df_cabang.count(), "| Menu =", df_menu.count(), "| Transaksi =", df_transaksi.count())


### Bagian B: TRANSFORM — Join & Kolom `total_penjualan`


In [ ]:
# Menggabungkan ketiga tabel
df_gabung = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                        .join(df_menu, on="menu_id", how="inner") \
                        .withColumn("total_penjualan", col("qty") * col("harga"))

df_gabung.show(5)


### Bagian C: TRANSFORM — Window Function: Top-2 Menu per Cabang


In [ ]:
# Agregasi total penjualan per kombinasi cabang dan menu
penjualan_menu_cabang = df_gabung.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("omset_menu")
)

# Window Function untuk mendapatkan Top-2 Menu
window_cabang = Window.partitionBy("nama_cabang").orderBy(desc("omset_menu"))

top2_menu = penjualan_menu_cabang.withColumn(
    "peringkat", row_number().over(window_cabang)
).filter(col("peringkat") <= 2)

print("Top 2 Menu Terlaris di Setiap Cabang:")
top2_menu.show()


### Bagian D: TRANSFORM — Spark SQL: Penjualan per Kepala Barista


In [ ]:
df_gabung.createOrReplaceTempView("kopi_nusantara")

kueri_barista = """
    SELECT 
        kepala_barista,
        COUNT(trx_id) AS total_transaksi,
        SUM(total_penjualan) AS omset_total
    FROM kopi_nusantara
    GROUP BY kepala_barista
    ORDER BY omset_total DESC
"""

spark.sql(kueri_barista).show()


### Bagian E: LOAD — Menyimpan ke Parquet Terpartisi


In [ ]:
output_tugas7 = "laporan_kopi_nusantara_parquet"

df_gabung.write.mode("overwrite").partitionBy("kategori_menu").parquet(output_tugas7)

# Verifikasi
df_verif = spark.read.parquet(output_tugas7)
print(f"Data tersimpan sukses. Jumlah baris: {df_verif.count()}")


### Bagian F: Ringkasan Eksekutif & Rekomendasi


In [ ]:
ringkasan_eksekutif = df_gabung.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_omset"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_transaksi")
).orderBy(desc("total_omset"))

ringkasan_eksekutif.show()


#### Rekomendasi Bisnis (Minimal 120 Kata):
Berdasarkan ringkasan eksekutif dan analisis granular di atas, berikut adalah rekomendasi strategis bagi pimpinan manajemen Kopi Nusantara:

1. **Evaluasi Performa Cabang**:
   Cabang dengan performa terbaik mencatatkan stabilitas volume transaksi dan omset tertinggi. Sementara itu, cabang dengan omset terendah membutuhkan perhatian segera pada jam operasional dan bauran produk (*product mix*).

2. **Strategi Menu & Promosi**:
   Berdasarkan temuan Window Function pada Bagian C, menu kategori Non-Kopi dan Makanan Ringan yang menduduki peringkat teratas membuktikan bahwa pelanggan kedai kopi sering kali datang untuk nongkrong atau mencari alternatif minuman manis. Di cabang dengan kinerja terlemah, manajemen sebaiknya meluncurkan program *bundling* antara minuman kopi unggulan dengan Top-1 makanan ringan terlaris dengan diskon khusus.

3. **Peran Kepala Barista**:
   Kepemimpinan kepala barista terbukti mempengaruhi konsistensi output transaksi. Manajemen perlu menerapkan program pelatihan silang (*cross-training*) antar-cabang agar standar pelayanan dan kecepatan penyajian di cabang terlemah dapat menyamai cabang teratas.
